#1차 시도 그러나 실패

In [ ]:
from pathlib import Path
import random
import math
import time
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# =========================================================
# 0. 경로 / 실행 옵션
# =========================================================
SAVEFILE_DIR = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2")
OUT_ROOT = SAVEFILE_DIR / "_circle_anchor_test"
RANDOM_SEED = 42
SAMPLES_PER_SUBFOLDER = 5
MAX_WORKERS = 8
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# =========================================================
# 1. 원 검출 파라미터
#    - 처음엔 이대로 돌리고, 필요하면 여기만 조절
# =========================================================
LEFT_ROI_RATIO = 0.35           # 왼쪽 몇 % 영역만 탐색할지
UPSCALE = 1.5                   # 작은 원 검출 안정화용
BLUR_KSIZE = 5
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)

# Hough circle 후보
HOUGH_DP = 1.2
HOUGH_MIN_DIST = 25
HOUGH_PARAM1 = 100
HOUGH_PARAM2 = 18
HOUGH_MIN_RADIUS = 6
HOUGH_MAX_RADIUS = 45

# contour 기반 보조 필터
CANNY1 = 40
CANNY2 = 120
MIN_AREA = 80
MAX_AREA = 6000
MIN_CIRCULARITY = 0.45
ELLIPSE_MAX_AXIS_RATIO = 1.8

# 성공 기준
SUCCESS_IF_AT_LEAST = 4         # 일단 4개 기준으로 성공률 계산
PARTIAL_IF_AT_LEAST = 3         # 3개면 partial

# 시각화
DRAW_RADIUS = 10
FONT = cv2.FONT_HERSHEY_SIMPLEX

In [ ]:
# =========================================================
# 2. 유틸
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def list_leaf_subfolders(root_dir):
    root_dir = Path(root_dir)
    all_dirs = [p for p in root_dir.rglob("*") if p.is_dir()]
    leaf_dirs = []
    for d in all_dirs:
        has_img = any(x.is_file() and x.suffix.lower() in IMAGE_EXTS for x in d.iterdir())
        has_subdir = any(x.is_dir() for x in d.iterdir())
        if has_img and not has_subdir:
            leaf_dirs.append(d)
    if not leaf_dirs:
        # 혹시 바로 아래가 이미지 폴더 구조가 아니어도, 이미지 있는 폴더 전부 수용
        fallback = []
        for d in all_dirs:
            has_img = any(x.is_file() and x.suffix.lower() in IMAGE_EXTS for x in d.iterdir())
            if has_img:
                fallback.append(d)
        leaf_dirs = fallback
    return sorted(set(leaf_dirs))


def sample_images_from_each_subfolder(root_dir, n_per_folder=5, seed=42):
    rng = random.Random(seed)
    subfolders = list_leaf_subfolders(root_dir)
    sampled = []

    for sub in subfolders:
        imgs = sorted([p for p in sub.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
        if not imgs:
            continue
        k = min(n_per_folder, len(imgs))
        chosen = rng.sample(imgs, k)
        for p in chosen:
            sampled.append({
                "subfolder": str(sub),
                "image_path": str(p),
                "image_name": p.name,
            })
    return pd.DataFrame(sampled)


def order_points_tl_tr_br_bl(points):
    pts = np.array(points, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    ordered = np.array([tl, tr, br, bl], dtype=np.float32)
    return ordered


def circularity_from_contour(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    if peri <= 0:
        return 0.0
    return float(4.0 * math.pi * area / (peri * peri))


def preprocess_gray(gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    out = clahe.apply(gray)
    if UPSCALE != 1.0:
        h, w = out.shape[:2]
        out = cv2.resize(out, (int(w * UPSCALE), int(h * UPSCALE)), interpolation=cv2.INTER_CUBIC)
    out = cv2.GaussianBlur(out, (BLUR_KSIZE, BLUR_KSIZE), 0)
    return out


def detect_circles_hough(proc_gray):
    circles = cv2.HoughCircles(
        proc_gray,
        cv2.HOUGH_GRADIENT,
        dp=HOUGH_DP,
        minDist=HOUGH_MIN_DIST,
        param1=HOUGH_PARAM1,
        param2=HOUGH_PARAM2,
        minRadius=HOUGH_MIN_RADIUS,
        maxRadius=HOUGH_MAX_RADIUS,
    )
    results = []
    if circles is None:
        return results
    circles = np.round(circles[0]).astype(int)
    for c in circles:
        x, y, r = int(c[0]), int(c[1]), int(c[2])
        results.append({
            "cx": x,
            "cy": y,
            "radius": r,
            "method": "hough",
            "score": float(r),
            "circularity": np.nan,
        })
    return results


def detect_circles_contour(proc_gray):
    edges = cv2.Canny(proc_gray, CANNY1, CANNY2)
    contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    results = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA or area > MAX_AREA:
            continue
        peri = cv2.arcLength(cnt, True)
        if peri <= 0:
            continue
        circ = circularity_from_contour(cnt)
        if circ < MIN_CIRCULARITY:
            continue
        if len(cnt) < 5:
            continue
        (cx, cy), (ma, mi), angle = cv2.fitEllipse(cnt)
        long_axis = max(ma, mi)
        short_axis = min(ma, mi)
        if short_axis <= 0:
            continue
        axis_ratio = long_axis / short_axis
        if axis_ratio > ELLIPSE_MAX_AXIS_RATIO:
            continue
        radius = 0.25 * (ma + mi)
        results.append({
            "cx": float(cx),
            "cy": float(cy),
            "radius": float(radius),
            "method": "contour",
            "score": float(area),
            "circularity": float(circ),
        })
    return results


def deduplicate_circles(candidates, dist_thresh=18):
    if not candidates:
        return []
    candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
    kept = []
    for c in candidates:
        ok = True
        for k in kept:
            d = ((c["cx"] - k["cx"]) ** 2 + (c["cy"] - k["cy"]) ** 2) ** 0.5
            if d < dist_thresh:
                ok = False
                break
        if ok:
            kept.append(c)
    return kept


def select_best_four(candidates, roi_w, roi_h):
    if len(candidates) <= 4:
        return candidates

    # 너무 오른쪽/아래로 튀는 후보 억제: 왼쪽 패널의 4원 가정
    scored = []
    for c in candidates:
        norm_x = c["cx"] / max(roi_w, 1)
        norm_y = c["cy"] / max(roi_h, 1)
        # 왼쪽일수록, 세로로 넓게 퍼질수록 선호
        score = c["score"] - 50 * norm_x
        scored.append((score, c))
    scored = sorted(scored, key=lambda x: x[0], reverse=True)
    top = [x[1] for x in scored[:8]]

    # x가 작은 순으로 먼저 압축 후, y 분포 고려해서 4개 선택
    top = sorted(top, key=lambda z: (z["cx"], z["cy"]))
    if len(top) <= 4:
        return top

    # 가장 왼쪽에 몰린 후보 위주 선택
    top = top[:6]
    top = sorted(top, key=lambda z: z["cy"])
    return top[:4]


def detect_four_circle_centers(image_bgr):
    h, w = image_bgr.shape[:2]
    roi_w = int(w * LEFT_ROI_RATIO)
    roi = image_bgr[:, :roi_w].copy()
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    proc = preprocess_gray(gray)

    hough = detect_circles_hough(proc)
    contour = detect_circles_contour(proc)
    candidates = hough + contour
    candidates = deduplicate_circles(candidates)
    candidates = select_best_four(candidates, proc.shape[1], proc.shape[0])

    # upscale 원복
    final_pts = []
    for c in candidates:
        final_pts.append({
            "cx": c["cx"] / UPSCALE,
            "cy": c["cy"] / UPSCALE,
            "radius": c["radius"] / UPSCALE,
            "method": c["method"],
            "score": c["score"],
            "circularity": c["circularity"],
        })

    final_pts = sorted(final_pts, key=lambda z: z["cy"])
    return roi, final_pts


def classify_detection(n_found):
    if n_found >= SUCCESS_IF_AT_LEAST:
        return "success"
    if n_found >= PARTIAL_IF_AT_LEAST:
        return "partial"
    return "fail"


def draw_overlay(image_bgr, roi_bgr, points, save_path):
    vis = image_bgr.copy()
    h, w = vis.shape[:2]
    roi_w = roi_bgr.shape[1]

    # ROI 박스
    cv2.rectangle(vis, (0, 0), (roi_w - 1, h - 1), (0, 255, 255), 2)

    # 검출점
    for i, p in enumerate(points, start=1):
        cx = int(round(p["cx"]))
        cy = int(round(p["cy"]))
        rr = int(round(max(4, p["radius"])))
        cv2.circle(vis, (cx, cy), rr, (0, 255, 0), 2)
        cv2.circle(vis, (cx, cy), 3, (0, 0, 255), -1)
        label = f"{i}"
        cv2.putText(vis, label, (cx + 6, cy - 6), FONT, 0.6, (255, 0, 0), 2, cv2.LINE_AA)

    # 4개면 순서 정렬해서 선 연결
    if len(points) >= 4:
        pts = np.array([[p["cx"], p["cy"]] for p in points[:4]], dtype=np.float32)
        ordered = order_points_tl_tr_br_bl(pts)
        ordered_i = ordered.astype(int)
        for i in range(4):
            p1 = tuple(ordered_i[i])
            p2 = tuple(ordered_i[(i + 1) % 4])
            cv2.line(vis, p1, p2, (255, 255, 0), 2)

    cv2.imwrite(str(save_path), vis)


def process_one_image(row, out_overlay_dir):
    image_path = Path(row["image_path"])
    subfolder = row["subfolder"]
    image_name = row["image_name"]

    result = {
        "subfolder": subfolder,
        "image_path": str(image_path),
        "image_name": image_name,
        "n_found": 0,
        "status": "fail",
        "p1_x": np.nan, "p1_y": np.nan,
        "p2_x": np.nan, "p2_y": np.nan,
        "p3_x": np.nan, "p3_y": np.nan,
        "p4_x": np.nan, "p4_y": np.nan,
        "methods": "",
        "notes": "",
    }

    img = cv2.imread(str(image_path))
    if img is None:
        result["notes"] = "image_read_fail"
        return result

    try:
        roi, points = detect_four_circle_centers(img)
        n_found = len(points)
        status = classify_detection(n_found)
        result["n_found"] = n_found
        result["status"] = status
        result["methods"] = ",".join([p["method"] for p in points])

        for i, p in enumerate(points[:4], start=1):
            result[f"p{i}_x"] = float(p["cx"])
            result[f"p{i}_y"] = float(p["cy"])

        overlay_name = image_path.stem + f"__{status}.jpg"
        save_path = Path(out_overlay_dir) / overlay_name
        draw_overlay(img, roi, points, save_path)

        if status == "partial":
            result["notes"] = "3_points_found_check_manually"
        elif status == "fail":
            result["notes"] = "less_than_3_points"

        return result

    except Exception as e:
        result["notes"] = f"exception: {str(e)[:200]}"
        return result


def summarize_results(df):
    total = len(df)
    success = int((df["status"] == "success").sum())
    partial = int((df["status"] == "partial").sum())
    fail = int((df["status"] == "fail").sum())

    return pd.DataFrame([
        {"metric": "total_images", "value": total},
        {"metric": "success_count", "value": success},
        {"metric": "partial_count", "value": partial},
        {"metric": "fail_count", "value": fail},
        {"metric": "success_rate", "value": round(success / total, 4) if total else 0},
        {"metric": "success_plus_partial_rate", "value": round((success + partial) / total, 4) if total else 0},
    ])

In [ ]:
# =========================================================
# 3. 메인 실행
# =========================================================
def main():
    t0 = time.time()
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    ensure_dir(OUT_ROOT)
    overlay_dir = OUT_ROOT / "overlays"
    ensure_dir(overlay_dir)

    print("[1] 하위 폴더별 랜덤 샘플 추출 중...")
    sample_df = sample_images_from_each_subfolder(
        SAVEFILE_DIR,
        n_per_folder=SAMPLES_PER_SUBFOLDER,
        seed=RANDOM_SEED,
    )

    sample_csv = OUT_ROOT / "sampled_images.csv"
    sample_df.to_csv(sample_csv, index=False, encoding="utf-8-sig")
    print(f"샘플 수: {len(sample_df):,}")
    print(f"샘플 목록 저장: {sample_csv}")

    if len(sample_df) == 0:
        print("이미지가 없습니다. 경로 확인 필요.")
        return

    print("[2] 원 4개 검출 테스트 중...")
    rows = sample_df.to_dict("records")
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_one_image, row, overlay_dir) for row in rows]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            results.append(fut.result())

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(["subfolder", "image_name"]).reset_index(drop=True)

    summary_df = summarize_results(result_df)
    by_folder_df = (
        result_df.groupby("subfolder")["status"]
        .value_counts(dropna=False)
        .unstack(fill_value=0)
        .reset_index()
    )
    by_folder_df["total"] = by_folder_df[[c for c in by_folder_df.columns if c in ["success", "partial", "fail"]]].sum(axis=1)
    by_folder_df["success_rate"] = np.where(by_folder_df["total"] > 0, by_folder_df.get("success", 0) / by_folder_df["total"], 0)
    by_folder_df["success_plus_partial_rate"] = np.where(
        by_folder_df["total"] > 0,
        (by_folder_df.get("success", 0) + by_folder_df.get("partial", 0)) / by_folder_df["total"],
        0,
    )

    excel_path = OUT_ROOT / "circle_anchor_test_report.xlsx"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        sample_df.to_excel(writer, sheet_name="sampled_images", index=False)
        result_df.to_excel(writer, sheet_name="per_image_results", index=False)
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        by_folder_df.to_excel(writer, sheet_name="by_folder", index=False)

    csv_path = OUT_ROOT / "per_image_results.csv"
    result_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    elapsed = time.time() - t0
    print("\n===== 완료 =====")
    print(f"엑셀: {excel_path}")
    print(f"CSV : {csv_path}")
    print(f"Overlay 폴더: {overlay_dir}")
    print(f"총 시간: {elapsed/60:.1f}분")
    print(summary_df)


if __name__ == "__main__":
    main()

# =========================================================
# 4. 메모
# =========================================================
# - 이 코드는 지금 단계에서 '원 4개가 기준점으로 안정적인지' 테스트용임.
# - affine 정렬은 이 다음 단계.
# - 넓게 보면 affine도 warp의 한 종류가 맞음.
#   다만 우리가 하려는 건 강한 perspective warp가 아니라,
#   약한 affine/정렬형 warp라고 보면 됨.
# - 즉 순서는:
#   랜덤샘플 추출 -> 원 4개 검출 성공률 확인 -> overlay 눈검사 ->
#   성공 샘플 대상으로 affine 정렬 시험
# =========================================================


[1] 하위 폴더별 랜덤 샘플 추출 중...
샘플 수: 100
샘플 목록 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test/sampled_images.csv
[2] 원 4개 검출 테스트 중...


  0%|          | 0/100 [00:00<?, ?it/s]


===== 완료 =====
엑셀: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test/circle_anchor_test_report.xlsx
CSV : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test/per_image_results.csv
Overlay 폴더: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test/overlays
총 시간: 0.5분
                      metric  value
0               total_images  100.0
1              success_count  100.0
2              partial_count    0.0
3                 fail_count    0.0
4               success_rate    1.0
5  success_plus_partial_rate    1.0


그러나 말만 성공이지 데이터는 완전한 실패였다. 4점이란 개념도 없고 원을 찾는다는 개념이 아예 없었다.

#실패해서 2차시도

In [ ]:
from pathlib import Path
import random
import math
import time
import itertools
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# =========================================================
# 0. 경로 / 실행 옵션
# =========================================================
SAVEFILE_DIR = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2")
OUT_ROOT = SAVEFILE_DIR / "_circle_anchor_test_v2"
RANDOM_SEED = 42
SAMPLES_PER_SUBFOLDER = 5
MAX_WORKERS = 8
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# =========================================================
# 1. 탐색 범위 / 샘플링
# =========================================================
LEFT_ROI_RATIO = 0.25           # 왼쪽 anchor 영역만 더 엄격하게 사용
UPSCALE = 1.5                   # 작은 hole 후보 검출 안정화

# =========================================================
# 2. 전처리 파라미터
# =========================================================
BLUR_KSIZE = 5
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)

# =========================================================
# 3. 검은 원 후보 찾기 파라미터
#    핵심: 원 '검출'보다 검은 hole '후보 수집' 단계
# =========================================================
THRESH_METHOD = "otsu_inv"
MIN_AREA = 120
MAX_AREA = 7000
MIN_CIRCULARITY = 0.35
MAX_AXIS_RATIO = 2.4
MIN_DARKNESS_DIFF = 12          # 내부가 주변보다 얼마나 더 어두워야 하는지
RING_MARGIN = 1.7               # 후보 주변 링 영역 밝기 비교용
DEDUP_DIST = 18                 # 중심점 중복 제거 거리
MIN_VISIBLE_RATE = 0.25         # 잘린 원 허용 최소치(보이는 bbox 면적 비율 대용)

# =========================================================
# 4. 조합 선택 규칙
#    핵심: 20x20 간격의 2x2 구조에 가까운 4개 조합을 고름
# =========================================================
MAX_CANDIDATES_FOR_COMB = 10
MIN_QUAD_AREA = 1500
MAX_QUAD_AREA = 120000
TOP_BOTTOM_Y_GAP_MIN = 8
DIST_RATIO_MIN = 0.45
DIST_RATIO_MAX = 2.20
ANGLE_PARALLEL_TOL_DEG = 28
FINAL_SCORE_MIN = 45.0          # 이 점수 이하면 fail

# =========================================================
# 5. 시각화
# =========================================================
FONT = cv2.FONT_HERSHEY_SIMPLEX

# =========================================================
# 6. 유틸
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def list_leaf_subfolders(root_dir):
    root_dir = Path(root_dir)
    all_dirs = [p for p in root_dir.rglob("*") if p.is_dir()]
    leaf_dirs = []
    for d in all_dirs:
        has_img = any(x.is_file() and x.suffix.lower() in IMAGE_EXTS for x in d.iterdir())
        has_subdir = any(x.is_dir() for x in d.iterdir())
        if has_img and not has_subdir:
            leaf_dirs.append(d)
    if not leaf_dirs:
        fallback = []
        for d in all_dirs:
            has_img = any(x.is_file() and x.suffix.lower() in IMAGE_EXTS for x in d.iterdir())
            if has_img:
                fallback.append(d)
        leaf_dirs = fallback
    return sorted(set(leaf_dirs))


def sample_images_from_each_subfolder(root_dir, n_per_folder=5, seed=42):
    rng = random.Random(seed)
    subfolders = list_leaf_subfolders(root_dir)
    sampled = []
    for sub in subfolders:
        imgs = sorted([p for p in sub.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
        if not imgs:
            continue
        k = min(n_per_folder, len(imgs))
        chosen = rng.sample(imgs, k)
        for p in chosen:
            sampled.append({
                "subfolder": str(sub),
                "image_path": str(p),
                "image_name": p.name,
            })
    return pd.DataFrame(sampled)


def angle_deg(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return math.degrees(math.atan2(dy, dx))


def angle_diff_abs(a, b):
    d = abs(a - b) % 180
    if d > 90:
        d = 180 - d
    return d


def dist(p1, p2):
    return float(((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2) ** 0.5)


def polygon_area(pts):
    pts = np.array(pts, dtype=np.float32)
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def order_points_tl_tr_br_bl(points):
    pts = np.array(points, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def circularity_from_contour(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    if peri <= 0:
        return 0.0
    return float(4.0 * math.pi * area / (peri * peri))


def preprocess_gray(gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    out = clahe.apply(gray)
    if UPSCALE != 1.0:
        h, w = out.shape[:2]
        out = cv2.resize(out, (int(w * UPSCALE), int(h * UPSCALE)), interpolation=cv2.INTER_CUBIC)
    out = cv2.GaussianBlur(out, (BLUR_KSIZE, BLUR_KSIZE), 0)
    return out


def make_binary_for_dark_holes(proc_gray):
    if THRESH_METHOD == "otsu_inv":
        _, bw = cv2.threshold(proc_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        bw = cv2.adaptiveThreshold(
            proc_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 31, 7
        )
    kernel = np.ones((3, 3), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=1)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=1)
    return bw


def compute_darkness_score(gray, cx, cy, radius):
    h, w = gray.shape[:2]
    yy, xx = np.indices((h, w))
    rr2 = (xx - cx) ** 2 + (yy - cy) ** 2
    inner = rr2 <= (radius ** 2)
    outer = (rr2 <= ((radius * RING_MARGIN) ** 2)) & (~inner)

    if inner.sum() < 20 or outer.sum() < 20:
        return None, None, 0.0

    inner_mean = float(gray[inner].mean())
    outer_mean = float(gray[outer].mean())
    diff = outer_mean - inner_mean
    return inner_mean, outer_mean, diff


def candidate_visible_rate(x, y, w, h, roi_w, roi_h):
    box_area = max(1, w * h)
    clipped_x1 = max(0, x)
    clipped_y1 = max(0, y)
    clipped_x2 = min(roi_w, x + w)
    clipped_y2 = min(roi_h, y + h)
    clipped_area = max(0, clipped_x2 - clipped_x1) * max(0, clipped_y2 - clipped_y1)
    return clipped_area / box_area


def detect_dark_hole_candidates(roi_bgr):
    gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    proc = preprocess_gray(gray)
    bw = make_binary_for_dark_holes(proc)

    contours, _ = cv2.findContours(bw, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    ph, pw = proc.shape[:2]

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA or area > MAX_AREA:
            continue
        peri = cv2.arcLength(cnt, True)
        if peri <= 0 or len(cnt) < 5:
            continue

        circ = circularity_from_contour(cnt)
        if circ < MIN_CIRCULARITY:
            continue

        (cx, cy), (ma, mi), _ = cv2.fitEllipse(cnt)
        major = max(ma, mi)
        minor = min(ma, mi)
        if minor <= 0:
            continue
        axis_ratio = major / minor
        if axis_ratio > MAX_AXIS_RATIO:
            continue

        radius = 0.25 * (ma + mi)
        x, y, w, h = cv2.boundingRect(cnt)
        visible_rate = candidate_visible_rate(x, y, w, h, pw, ph)
        if visible_rate < MIN_VISIBLE_RATE:
            continue

        inner_mean, outer_mean, dark_diff = compute_darkness_score(proc, cx, cy, radius)
        if dark_diff is None or dark_diff < MIN_DARKNESS_DIFF:
            continue

        # ROI 높이 기준
        roi_h = ph / UPSCALE

        # 중심 y 기준 필터
        cy_real = cy / UPSCALE

        if cy_real < roi_h * TOP_CUT_RATIO:
            continue

        if cy_real > roi_h * (1 - BOTTOM_CUT_RATIO):
            continue

        candidates.append({
            "cx": float(cx / UPSCALE),
            "cy": float(cy / UPSCALE),
            "radius": float(radius / UPSCALE),
            "circularity": float(circ),
            "axis_ratio": float(axis_ratio),
            "inner_mean": float(inner_mean),
            "outer_mean": float(outer_mean),
            "dark_diff": float(dark_diff),
            "visible_rate": float(visible_rate),
            "score_individual": float(dark_diff + 20 * circ - 5 * (axis_ratio - 1.0) + 8 * visible_rate),
        })

    return deduplicate_candidates(candidates, DEDUP_DIST)


def deduplicate_candidates(candidates, dist_thresh=18):
    if not candidates:
        return []
    candidates = sorted(candidates, key=lambda x: x["score_individual"], reverse=True)
    kept = []
    for c in candidates:
        ok = True
        for k in kept:
            if dist((c["cx"], c["cy"]), (k["cx"], k["cy"])) < dist_thresh:
                ok = False
                break
        if ok:
            kept.append(c)
    return kept


def score_four_point_combination(combo, roi_w, roi_h):
    pts = np.array([[c["cx"], c["cy"]] for c in combo], dtype=np.float32)
    ordered = order_points_tl_tr_br_bl(pts)
    tl, tr, br, bl = ordered

    quad_area = polygon_area(ordered)
    if quad_area < MIN_QUAD_AREA or quad_area > MAX_QUAD_AREA:
        return None

    top_w = dist(tl, tr)
    bottom_w = dist(bl, br)
    left_h = dist(tl, bl)
    right_h = dist(tr, br)
    diag1 = dist(tl, br)
    diag2 = dist(tr, bl)

    if min(top_w, bottom_w, left_h, right_h, diag1, diag2) <= 1:
        return None

    top_y = 0.5 * (tl[1] + tr[1])
    bottom_y = 0.5 * (bl[1] + br[1])
    if (bottom_y - top_y) < TOP_BOTTOM_Y_GAP_MIN:
        return None

    width_ratio = max(top_w, bottom_w) / min(top_w, bottom_w)
    height_ratio = max(left_h, right_h) / min(left_h, right_h)
    diag_ratio = max(diag1, diag2) / min(diag1, diag2)

    if not (DIST_RATIO_MIN <= width_ratio <= DIST_RATIO_MAX):
        return None
    if not (DIST_RATIO_MIN <= height_ratio <= DIST_RATIO_MAX):
        return None
    if not (DIST_RATIO_MIN <= diag_ratio <= DIST_RATIO_MAX):
        return None

    ang_top = angle_deg(tl, tr)
    ang_bottom = angle_deg(bl, br)
    ang_left = angle_deg(tl, bl)
    ang_right = angle_deg(tr, br)
    diff_h = angle_diff_abs(ang_top, ang_bottom)
    diff_v = angle_diff_abs(ang_left, ang_right)

    if diff_h > ANGLE_PARALLEL_TOL_DEG:
        return None
    if diff_v > ANGLE_PARALLEL_TOL_DEG:
        return None

    centroid_x = float(ordered[:, 0].mean())
    centroid_y = float(ordered[:, 1].mean())
    x_pos_ratio = centroid_x / max(roi_w, 1)
    y_pos_ratio = centroid_y / max(roi_h, 1)

    indiv = sum(c["score_individual"] for c in combo)
    radii = [c["radius"] for c in combo]
    rad_ratio = max(radii) / max(min(radii), 1e-6)

    score = 0.0
    score += indiv
    score += 20 * (1 / width_ratio)
    score += 20 * (1 / height_ratio)
    score += 12 * (1 / diag_ratio)
    score += 12 * (1 - diff_h / max(ANGLE_PARALLEL_TOL_DEG, 1))
    score += 12 * (1 - diff_v / max(ANGLE_PARALLEL_TOL_DEG, 1))
    score += 10 * max(0, 1 - x_pos_ratio)
    score += 4 * max(0, 1 - abs(y_pos_ratio - 0.35))
    score -= 8 * max(0, rad_ratio - 1.6)

    return {
        "ordered": ordered,
        "score": float(score),
        "quad_area": float(quad_area),
        "width_ratio": float(width_ratio),
        "height_ratio": float(height_ratio),
        "diag_ratio": float(diag_ratio),
        "diff_h": float(diff_h),
        "diff_v": float(diff_v),
    }


def choose_best_anchor_four(candidates, roi_w, roi_h):
    if len(candidates) < 4:
        return None, candidates

    candidates = sorted(candidates, key=lambda x: x["score_individual"], reverse=True)
    candidates = candidates[:MAX_CANDIDATES_FOR_COMB]

    best = None
    for combo in itertools.combinations(candidates, 4):
        scored = score_four_point_combination(combo, roi_w, roi_h)
        if scored is None:
            continue
        if best is None or scored["score"] > best["score"]:
            best = scored.copy()
            best["combo"] = combo

    if best is None or best["score"] < FINAL_SCORE_MIN:
        return None, candidates

    ordered_pts = best["ordered"]
    chosen = []
    for pt in ordered_pts:
        matched = min(best["combo"], key=lambda c: dist((float(pt[0]), float(pt[1])), (c["cx"], c["cy"])))
        chosen.append(matched)

    return {
        "chosen": chosen,
        "ordered": ordered_pts,
        "score": best["score"],
        "quad_area": best["quad_area"],
        "width_ratio": best["width_ratio"],
        "height_ratio": best["height_ratio"],
        "diag_ratio": best["diag_ratio"],
        "diff_h": best["diff_h"],
        "diff_v": best["diff_v"],
    }, candidates


def classify_detection(best_anchor):
    return "success" if best_anchor is not None else "fail"


def draw_overlay(image_bgr, roi_bgr, all_candidates, best_anchor, save_path):
    vis = image_bgr.copy()
    h, _ = vis.shape[:2]
    roi_w = roi_bgr.shape[1]
    cv2.rectangle(vis, (0, 0), (roi_w - 1, h - 1), (0, 255, 255), 2)

    for c in all_candidates:
        cx = int(round(c["cx"]))
        cy = int(round(c["cy"]))
        rr = int(round(max(4, c["radius"])))
        cv2.circle(vis, (cx, cy), rr, (180, 180, 180), 1)
        cv2.circle(vis, (cx, cy), 2, (0, 255, 255), -1)

    if best_anchor is not None:
        chosen = best_anchor["chosen"]
        ordered = best_anchor["ordered"].astype(int)
        for i, p in enumerate(chosen, start=1):
            cx = int(round(p["cx"]))
            cy = int(round(p["cy"]))
            rr = int(round(max(4, p["radius"])))
            cv2.circle(vis, (cx, cy), rr, (0, 255, 0), 2)
            cv2.circle(vis, (cx, cy), 3, (0, 0, 255), -1)
            cv2.putText(vis, str(i), (cx + 6, cy - 6), FONT, 0.65, (255, 0, 0), 2, cv2.LINE_AA)

        for i in range(4):
            p1 = tuple(ordered[i])
            p2 = tuple(ordered[(i + 1) % 4])
            cv2.line(vis, p1, p2, (255, 255, 0), 2)

        txt = f"score={best_anchor['score']:.1f} wr={best_anchor['width_ratio']:.2f} hr={best_anchor['height_ratio']:.2f}"
        cv2.putText(vis, txt, (10, h - 18), FONT, 0.55, (0, 255, 0), 2, cv2.LINE_AA)
    else:
        cv2.putText(vis, "FAIL: no valid 2x2 anchor set", (10, h - 18), FONT, 0.55, (0, 0, 255), 2, cv2.LINE_AA)

    cv2.imwrite(str(save_path), vis)


def process_one_image(row, out_overlay_dir):
    image_path = Path(row["image_path"])
    result = {
        "subfolder": row["subfolder"],
        "image_path": str(image_path),
        "image_name": row["image_name"],
        "n_candidates": 0,
        "status": "fail",
        "anchor_score": np.nan,
        "width_ratio": np.nan,
        "height_ratio": np.nan,
        "diag_ratio": np.nan,
        "parallel_h_diff_deg": np.nan,
        "parallel_v_diff_deg": np.nan,
        "p1_x": np.nan, "p1_y": np.nan,
        "p2_x": np.nan, "p2_y": np.nan,
        "p3_x": np.nan, "p3_y": np.nan,
        "p4_x": np.nan, "p4_y": np.nan,
        "notes": "",
    }

    img = cv2.imread(str(image_path))
    if img is None:
        result["notes"] = "image_read_fail"
        return result

    try:
        h, w = img.shape[:2]
        roi_w = int(w * LEFT_ROI_RATIO)
        roi = img[:, :roi_w].copy()

        candidates = detect_dark_hole_candidates(roi)
        result["n_candidates"] = len(candidates)
        best_anchor, used_candidates = choose_best_anchor_four(candidates, roi_w, h)
        result["status"] = classify_detection(best_anchor)

        if best_anchor is not None:
            result["anchor_score"] = float(best_anchor["score"])
            result["width_ratio"] = float(best_anchor["width_ratio"])
            result["height_ratio"] = float(best_anchor["height_ratio"])
            result["diag_ratio"] = float(best_anchor["diag_ratio"])
            result["parallel_h_diff_deg"] = float(best_anchor["diff_h"])
            result["parallel_v_diff_deg"] = float(best_anchor["diff_v"])
            ordered = best_anchor["ordered"]
            for i in range(4):
                result[f"p{i+1}_x"] = float(ordered[i][0])
                result[f"p{i+1}_y"] = float(ordered[i][1])
        else:
            result["notes"] = "no_valid_2x2_anchor_set"

        overlay_name = image_path.stem + f"__{result['status']}.jpg"
        save_path = Path(out_overlay_dir) / overlay_name
        draw_overlay(img, roi, used_candidates, best_anchor, save_path)
        return result

    except Exception as e:
        result["notes"] = f"exception: {str(e)[:200]}"
        return result


def summarize_results(df):
    total = len(df)
    success = int((df["status"] == "success").sum())
    fail = int((df["status"] == "fail").sum())
    return pd.DataFrame([
        {"metric": "total_images", "value": total},
        {"metric": "success_count", "value": success},
        {"metric": "fail_count", "value": fail},
        {"metric": "success_rate", "value": round(success / total, 4) if total else 0},
    ])


def main():
    t0 = time.time()
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    ensure_dir(OUT_ROOT)
    overlay_dir = OUT_ROOT / "overlays"
    ensure_dir(overlay_dir)

    print("[1] 하위 폴더별 랜덤 샘플 추출 중...")
    sample_df = sample_images_from_each_subfolder(
        SAVEFILE_DIR,
        n_per_folder=SAMPLES_PER_SUBFOLDER,
        seed=RANDOM_SEED,
    )
    sample_csv = OUT_ROOT / "sampled_images.csv"
    sample_df.to_csv(sample_csv, index=False, encoding="utf-8-sig")
    print(f"샘플 수: {len(sample_df):,}")

    if len(sample_df) == 0:
        print("이미지가 없습니다. 경로 확인 필요.")
        return

    print("[2] 검은 hole 후보 -> 2x2 anchor 조합 선택 테스트 중...")
    rows = sample_df.to_dict("records")
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_one_image, row, overlay_dir) for row in rows]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            results.append(fut.result())

    result_df = pd.DataFrame(results).sort_values(["subfolder", "image_name"]).reset_index(drop=True)
    summary_df = summarize_results(result_df)
    by_folder_df = (
        result_df.groupby("subfolder")["status"]
        .value_counts(dropna=False)
        .unstack(fill_value=0)
        .reset_index()
    )
    by_folder_df["total"] = by_folder_df[[c for c in by_folder_df.columns if c in ["success", "fail"]]].sum(axis=1)
    by_folder_df["success_rate"] = np.where(by_folder_df["total"] > 0, by_folder_df.get("success", 0) / by_folder_df["total"], 0)

    excel_path = OUT_ROOT / "circle_anchor_test_report_v2.xlsx"
    csv_path = OUT_ROOT / "per_image_results_v2.csv"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        sample_df.to_excel(writer, sheet_name="sampled_images", index=False)
        result_df.to_excel(writer, sheet_name="per_image_results", index=False)
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        by_folder_df.to_excel(writer, sheet_name="by_folder", index=False)
    result_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    elapsed = time.time() - t0
    print("\n===== 완료 =====")
    print(f"엑셀: {excel_path}")
    print(f"CSV : {csv_path}")
    print(f"Overlay 폴더: {overlay_dir}")
    print(f"총 시간: {elapsed/60:.1f}분")
    print(summary_df)


if __name__ == "__main__":
    main()

# =========================================================
# 메모
# =========================================================
# v2 핵심 변화
# 1) 단순 원 4개 찾기 -> 검은 hole 후보를 넓게 수집
# 2) 후보 4개 조합 중 2x2 anchor 구조를 가장 잘 만족하는 조합 선택
# 3) success는 '4개 있음'이 아니라 '기하 규칙 통과'일 때만 인정
# 4) 일부 잘린 원은 후보에는 포함 가능하지만, 조합 규칙까지 맞아야 최종 통과
# =========================================================


[1] 하위 폴더별 랜덤 샘플 추출 중...
샘플 수: 110
[2] 검은 hole 후보 -> 2x2 anchor 조합 선택 테스트 중...


  0%|          | 0/110 [00:00<?, ?it/s]


===== 완료 =====
엑셀: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_v2/circle_anchor_test_report_v2.xlsx
CSV : /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_v2/per_image_results_v2.csv
Overlay 폴더: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_v2/overlays
총 시간: 0.1분
          metric  value
0   total_images  110.0
1  success_count    0.0
2     fail_count  110.0
3   success_rate    0.0


어느정돈 성공을 했다. 77%의 성공률과 그중 26개에대해서는 완벽한 평행사변형을 찾아냈다. 그래서 최종 적용을 위한 마지막 3번째 4점찾기 코드로 들어간다. 성공한 것들을 벤치마킹해서 보정하는것이다.

#4점찾기 ver 3

In [ ]:
from pathlib import Path
import random
import math
import time
import itertools
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# =========================================================
# 0. 경로 / 실행 옵션
# =========================================================
SAVEFILE_DIR = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2")
V2_RESULT_CSV = SAVEFILE_DIR / "_circle_anchor_test_v2" / "per_image_results_v2._1.csv" # Use .csv as per discussion
OUT_ROOT = SAVEFILE_DIR / "_circle_anchor_test_final"

RANDOM_SEED = 42
MAX_WORKERS = 8
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"} # Ensure mutable set

# =========================================================
# 1. ROI / 전처리 / 후보 검출
# =========================================================
LEFT_ROI_RATIO = 0.28
UPSCALE = 1.5
BLUR_KSIZE = 5
CLAHE_CLIP = 2.0
CLAHE_TILE = (8, 8)
TOP_CUT_RATIO = 0.05
BOTTOM_CUT_RATIO = 0.05

THRESH_METHOD = "otsu_inv"
MIN_AREA = 120
MAX_AREA = 7000
MIN_CIRCULARITY = 0.35
MAX_AXIS_RATIO = 2.4
MIN_DARKNESS_DIFF = 12
RING_MARGIN = 1.7
DEDUP_DIST = 18
MIN_VISIBLE_RATE = 0.25

# =========================================================
# 2. template-driven 보정 규칙
#    핵심: 검은 원 4개로 사각형 만들기 X
#         good template를 먼저 놓고, 주변 후보를 그 template에 붙이기 O
# =========================================================
GOOD_NOTE_VALUE = "ㅇ"
PARTIAL_MATCH_DIST_RATIO = 0.16       # template 점 주변에서 후보를 매칭할 허용 반경
MAX_TEMPLATE_SHIFT_RATIO = 0.08       # 전체 template를 약간 이동시키며 최적화
TEMPLATE_SHIFT_STEPS = 5              # -2,-1,0,1,2 step
REQUIRE_MATCH_FOR_CORRECTED = 2       # 최소 2점 이상 맞으면 corrected, 아니면 template_only

FONT = cv2.FONT_HERSHEY_SIMPLEX

# =========================================================
# 유틸
# =========================================================
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def list_image_files_recursive(root_dir):
    root_dir = Path(root_dir)
    return sorted([p for p in root_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])


def dist(p1, p2):
    return float(np.linalg.norm(np.array(p1) - np.array(p2)))

def slope(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return math.degrees(math.atan2(dy, dx))


def order_points_tl_tr_br_bl(points):
    pts = np.array(points, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)
    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype=np.float32)


def circularity_from_contour(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    if peri <= 0:
        return 0.0
    return float(4.0 * math.pi * area / (peri * peri))


def preprocess_gray(gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    out = clahe.apply(gray)
    if UPSCALE != 1.0:
        h, w = out.shape[:2]
        out = cv2.resize(out, (int(w * UPSCALE), int(h * UPSCALE)), interpolation=cv2.INTER_CUBIC)
    out = cv2.GaussianBlur(out, (BLUR_KSIZE, BLUR_KSIZE), 0)
    return out


def make_binary_for_dark_holes(proc_gray):
    if THRESH_METHOD == "otsu_inv":
        _, bw = cv2.threshold(proc_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        bw = cv2.adaptiveThreshold(
            proc_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 31, 7
        )
    kernel = np.ones((3, 3), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel, iterations=1)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=1)
    return bw


def compute_darkness_score(gray, cx, cy, radius):
    h, w = gray.shape[:2]
    yy, xx = np.indices((h, w))
    rr2 = (xx - cx) ** 2 + (yy - cy) ** 2
    inner = rr2 <= (radius ** 2)
    outer = (rr2 <= ((radius * RING_MARGIN) ** 2)) & (~inner)
    if inner.sum() < 20 or outer.sum() < 20:
        return None, None, 0.0
    inner_mean = float(gray[inner].mean())
    outer_mean = float(gray[outer].mean())
    diff = outer_mean - inner_mean
    return inner_mean, outer_mean, diff


def candidate_visible_rate(x, y, w, h, roi_w, roi_h):
    box_area = max(1, w * h)
    clipped_x1 = max(0, x)
    clipped_y1 = max(0, y)
    clipped_x2 = min(roi_w, x + w)
    clipped_y2 = min(roi_h, y + h)
    clipped_area = max(0, clipped_x2 - clipped_x1) * max(0, clipped_y2 - clipped_y1)
    return clipped_area / box_area


def deduplicate_candidates(candidates, dist_thresh=18):
    if not candidates:
        return []
    candidates = sorted(candidates, key=lambda x: x["score_individual"], reverse=True)
    kept = []
    for c in candidates:
        ok = True
        for k in kept:
            if dist((c["cx"], c["cy"]), (k["cx"], k["cy"])) < dist_thresh:
                ok = False
                break
        if ok:
            kept.append(c)
    return kept


def detect_dark_hole_candidates(roi_bgr):
    gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    proc = preprocess_gray(gray)
    bw = make_binary_for_dark_holes(proc)
    contours, _ = cv2.findContours(bw, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    ph, pw = proc.shape[:2]

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < MIN_AREA or area > MAX_AREA:
            continue
        peri = cv2.arcLength(cnt, True)
        if peri <= 0 or len(cnt) < 5:
            continue

        circ = circularity_from_contour(cnt)
        if circ < MIN_CIRCULARITY:
            continue

        (cx, cy), (ma, mi), _ = cv2.fitEllipse(cnt)
        major = max(ma, mi)
        minor = min(ma, mi)
        if minor <= 0:
            continue
        axis_ratio = major / minor
        if axis_ratio > MAX_AXIS_RATIO:
            continue

        radius = 0.25 * (ma + mi)
        x, y, w, h = cv2.boundingRect(cnt)
        visible_rate = candidate_visible_rate(x, y, w, h, pw, ph)
        if visible_rate < MIN_VISIBLE_RATE:
            continue

        inner_mean, outer_mean, dark_diff = compute_darkness_score(proc, cx, cy, radius)
        if dark_diff is None or dark_diff < MIN_DARKNESS_DIFF:
            continue

        roi_h = ph / UPSCALE
        cy_real = cy / UPSCALE
        if cy_real < roi_h * TOP_CUT_RATIO:
            continue
        if cy_real > roi_h * (1 - BOTTOM_CUT_RATIO):
            continue

        candidates.append({
            "cx": float(cx / UPSCALE),
            "cy": float(cy / UPSCALE),
            "radius": float(radius / UPSCALE),
            "circularity": float(circ),
            "axis_ratio": float(axis_ratio),
            "inner_mean": float(inner_mean),
            "outer_mean": float(outer_mean),
            "dark_diff": float(dark_diff),
            "visible_rate": float(visible_rate),
            "score_individual": float(dark_diff + 20 * circ - 5 * (axis_ratio - 1.0) + 8 * visible_rate),
        })

    return deduplicate_candidates(candidates, DEDUP_DIST)

# =========================================================
# template 생성 / 적용
# =========================================================
def load_good_template_points(csv_path: Path, roi_ratio: float):
    if not csv_path.exists():
        raise FileNotFoundError(f"v2 결과 CSV가 없습니다: {csv_path}")

    df = pd.read_csv(csv_path)
    if "notes" not in df.columns:
        raise ValueError("v2 CSV에 notes 컬럼이 없습니다. good 케이스 표시가 필요합니다.")

    good = df[df["notes"].astype(str) == GOOD_NOTE_VALUE].copy()
    if len(good) == 0:
        raise ValueError("notes=='ㅇ' 인 good 케이스가 없습니다.")

    req_cols = [f"p{i}_{xy}" for i in range(1, 5) for xy in ["x", "y"]]
    miss = [c for c in req_cols if c not in good.columns]
    if miss:
        raise ValueError(f"다음 컬럼이 없습니다: {miss}")

    templates = []
    for _, row in good.iterrows():
        img_path = Path(row["image_path"])
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        roi_w = int(w * roi_ratio)
        pts = np.array([
            [row["p1_x"], row["p1_y"]],
            [row["p2_x"], row["p2_y"]],
            [row["p3_x"], row["p3_y"]],
            [row["p4_x"], row["p4_y"]],
        ], dtype=np.float32)
        pts = order_points_tl_tr_br_bl(pts)
        norm = pts.copy()
        norm[:, 0] = norm[:, 0] / max(roi_w, 1)
        norm[:, 1] = norm[:, 1] / max(h, 1)
        templates.append(norm)

    if len(templates) == 0:
        raise ValueError("good 케이스에서 template를 만들 수 없습니다.")

    arr = np.stack(templates, axis=0)
    mean_template = arr.mean(axis=0)
    std_template = arr.std(axis=0)
    return mean_template, std_template, len(templates)


def template_to_image_points(template_norm, roi_w, img_h, dx=0.0, dy=0.0):
    pts = np.array(template_norm, dtype=np.float32).copy()
    pts[:, 0] *= roi_w
    pts[:, 1] *= img_h
    pts[:, 0] += dx
    pts[:, 1] += dy
    return pts


def match_candidates_to_template(candidates, template_pts, roi_w, img_h):
    if len(candidates) == 0:
        return [], 0.0 # Return empty list and 0.0 for total_dist
    diag = math.hypot(roi_w, img_h)
    thr = PARTIAL_MATCH_DIST_RATIO * diag

    used = set()
    matched = []
    total_dist = 0.0
    for i, tp in enumerate(template_pts):
        best_j = None
        best_d = None
        for j, c in enumerate(candidates):
            if j in used:
                continue
            d = dist((float(tp[0]), float(tp[1])), (c["cx"], c["cy"]))
            if best_d is None or d < best_d:
                best_d = d
                best_j = j
        if best_j is not None and best_d is not None and best_d <= thr:
            used.add(best_j)
            matched.append((i, candidates[best_j], best_d))
            total_dist += best_d
    return matched, total_dist


def find_best_shifted_template(candidates, template_norm, roi_w, img_h):
    dx_step = roi_w * MAX_TEMPLATE_SHIFT_RATIO / max(TEMPLATE_SHIFT_STEPS - 1, 1)
    dy_step = img_h * MAX_TEMPLATE_SHIFT_RATIO / max(TEMPLATE_SHIFT_STEPS - 1, 1)

    offsets = []
    half = TEMPLATE_SHIFT_STEPS // 2
    for ix in range(-half, half + 1):
        for iy in range(-half, half + 1):
            offsets.append((ix * dx_step, iy * dy_step))

    best = None
    for dx, dy in offsets:
        template_pts = template_to_image_points(template_norm, roi_w, img_h, dx=dx, dy=dy)
        matched, total_dist = match_candidates_to_template(candidates, template_pts, roi_w, img_h)
        n = len(matched)
        avg_dist = total_dist / max(n, 1)
        # match 많이 될수록 우선, 거리 짧을수록 우선
        score = n * 1000 - avg_dist
        if (best is None) or (score > best["score"]):
            best = {
                "score": score,
                "dx": dx,
                "dy": dy,
                "template_pts": template_pts,
                "matched": matched,
                "matched_count": n,
                "avg_dist": avg_dist,
            }
    return best


def corrected_points_from_template(template_pts, matched_list):
    final_pts = template_pts.copy().astype(np.float32)
    for idx, cand, _ in matched_list:
        final_pts[idx, 0] = cand["cx"]
        final_pts[idx, 1] = cand["cy"]
    # 마지막엔 무조건 template 구조 순서로 정리
    final_pts = order_points_tl_tr_br_bl(final_pts)
    return final_pts

# =========================================================
# 시각화
# =========================================================
def draw_overlay(image_bgr, roi_bgr, all_candidates, final_pts, source, matched_count, save_path):
    vis = image_bgr.copy()
    h, _ = vis.shape[:2]
    roi_w = roi_bgr.shape[1]
    cv2.rectangle(vis, (0, 0), (roi_w - 1, h - 1), (0, 255, 255), 2)

    for c in all_candidates:
        cx = int(round(c["cx"]))
        cy = int(round(c["cy"]))
        rr = int(round(max(4, c["radius"])))
        cv2.circle(vis, (cx, cy), rr, (180, 180, 180), 1)
        cv2.circle(vis, (cx, cy), 2, (0, 255, 255), -1)

    pts = np.array(final_pts, dtype=np.float32)
    pts_i = pts.astype(int)
    for i, p in enumerate(pts_i, start=1):
        cv2.circle(vis, tuple(p), 10, (0, 255, 0), 2)
        cv2.circle(vis, tuple(p), 3, (0, 0, 255), -1)
        cv2.putText(vis, str(i), (p[0] + 6, p[1] - 6), FONT, 0.65, (255, 0, 0), 2, cv2.LINE_AA)
    for i in range(4):
        cv2.line(vis, tuple(pts_i[i]), tuple(pts_i[(i + 1) % 4]), (255, 255, 0), 2)

    color = (0, 255, 0) if source == "corrected" else (0, 0, 255)
    txt = f"source={source} matched={matched_count}"
    cv2.putText(vis, txt, (10, h - 18), FONT, 0.55, color, 2, cv2.LINE_AA)
    cv2.imwrite(str(save_path), vis)

# =========================================================
# 메인 처리
# =========================================================
def process_one_image(row, template_norm):
    image_path = Path(row["image_path"])
    out = {
        "subfolder": row["subfolder"],
        "image_path": str(image_path),
        "image_name": row["image_name"],
        "n_candidates": 0,
        "source": "template_only",
        "matched_count": 0,
        "avg_match_dist": np.nan,
        "template_dx": np.nan,
        "template_dy": np.nan,
        "notes": np.nan,
        "p1_x": np.nan, "p1_y": np.nan,
        "p2_x": np.nan, "p2_y": np.nan,
        "p3_x": np.nan, "p3_y": np.nan,
        "p4_x": np.nan, "p4_y": np.nan,
    }

    img = cv2.imread(str(image_path))
    if img is None:
        out["notes"] = "image_read_fail"
        return out, None, None, None, None

    h, w = img.shape[:2]
    roi_w = int(w * LEFT_ROI_RATIO)
    roi = img[:, :roi_w].copy()

    candidates = detect_dark_hole_candidates(roi)
    out["n_candidates"] = len(candidates)

    best = find_best_shifted_template(candidates, template_norm, roi_w, h)
    template_pts = best["template_pts"]
    matched = best["matched"]
    matched_count = best["matched_count"]
    avg_dist = best["avg_dist"]
    dx = best["dx"]
    dy = best["dy"]

    if matched_count >= REQUIRE_MATCH_FOR_CORRECTED:
        final_pts = corrected_points_from_template(template_pts, matched)
        source = "corrected"
        notes = f"template_corrected_with_{matched_count}"
    else:
        final_pts = order_points_tl_tr_br_bl(template_pts)
        source = "template_only"
        notes = "template_only"

    out["source"] = source
    out["matched_count"] = matched_count
    out["avg_match_dist"] = avg_dist
    out["template_dx"] = dx
    out["template_dy"] = dy
    out["notes"] = notes

    # Unpack final_pts for QC calculations
    p1, p2, p3, p4 = final_pts

    # ===== QC 계산 =====
    top_w = dist(p1, p2)
    bottom_w = dist(p4, p3)
    left_h = dist(p1, p4)
    right_h = dist(p2, p3)

    diag1 = dist(p1, p3)
    diag2 = dist(p2, p4)

    top_slope = slope(p1, p2)
    bottom_slope = slope(p4, p3)
    left_slope = slope(p1, p4)
    right_slope = slope(p2, p3)

    # ratio
    width_ratio = top_w / (bottom_w + 1e-6)
    height_ratio = left_h / (right_h + 1e-6)
    diag_ratio = diag1 / (diag2 + 1e-6)

    # area
    quad_area = 0.5 * abs(
        p1[0]*p2[1] + p2[0]*p3[1] + p3[0]*p4[1] + p4[0]*p1[1]
        - p2[0]*p1[1] - p3[0]*p2[1] - p4[0]*p3[1] - p1[0]*p4[1]
    )

    # centroid
    cx = np.mean([p1[0], p2[0], p3[0], p4[0]])
    cy = np.mean([p1[1], p2[1], p3[1], p4[1]])

    out.update({
        "quad_width_top": top_w,
        "quad_width_bottom": bottom_w,
        "quad_height_left": left_h,
        "quad_height_right": right_h,
        "quad_diag_1": diag1,
        "quad_diag_2": diag2,
        "top_slope_deg": top_slope,
        "bottom_slope_deg": bottom_slope,
        "left_slope_deg": left_slope,
        "right_slope_deg": right_slope,
        "width_ratio": width_ratio,
        "height_ratio": height_ratio,
        "diag_ratio": diag_ratio,
        "quad_area": quad_area,
        "centroid_x": cx,
        "centroid_y": cy,
    })

    for i in range(4):
        out[f"p{i+1}_x"] = float(final_pts[i][0])
        out[f"p{i+1}_y"] = float(final_pts[i][1])

    return out, img, roi, candidates, final_pts # Return final_pts for overlay


def main():
    t0 = time.time()
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    ensure_dir(OUT_ROOT)
    overlay_dir = OUT_ROOT / "overlays"
    ensure_dir(overlay_dir)

    print("[1] good 케이스(notes=='ㅇ')로 template 생성 중...")
    template_norm, template_std, n_good = load_good_template_points(V2_RESULT_CSV, LEFT_ROI_RATIO)
    template_df = pd.DataFrame({
        "point": ["p1", "p2", "p3", "p4"],
        "x_mean_norm": template_norm[:, 0],
        "y_mean_norm": template_norm[:, 1],
        "x_std_norm": template_std[:, 0],
        "y_std_norm": template_std[:, 1],
    })
    template_df.to_csv(OUT_ROOT / "template_summary.csv", index=False, encoding="utf-8-sig")
    print(f"good 케이스 수: {n_good}")

    print("[2] 전체 이미지 목록 수집 중...")
    image_paths = list_image_files_recursive(SAVEFILE_DIR)
    # 결과 폴더 내부는 제외
    image_paths = [p for p in image_paths if OUT_ROOT.name not in str(p) and "_circle_anchor_test_v2" not in str(p) and "_circle_anchor_test" not in str(p)]
    print(f"전체 이미지 수: {len(image_paths):,}")

    if len(image_paths) == 0:
        print("이미지가 없습니다. 경로 확인 필요.")
        return

    # Prepare DataFrame for all images
    all_images_data = []
    for p in image_paths:
        all_images_data.append({
            "subfolder": str(p.parent),
            "image_path": str(p),
            "image_name": p.name,
        })
    all_images_df = pd.DataFrame(all_images_data)

    print("[3] template-driven 방식으로 anchor 4점 생성 중...")
    rows = all_images_df.to_dict("records")
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = [ex.submit(process_one_image, row, template_norm) for row in rows]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            out, img, roi, candidates, final_pts_for_overlay = fut.result()
            results.append(out)
            if img is not None:
                overlay_name = Path(out["image_name"]).stem + f"__{out['source']}.jpg"
                save_path = overlay_dir / overlay_name
                draw_overlay(img, roi, candidates if candidates is not None else [], final_pts_for_overlay, out["source"], out["matched_count"], save_path)

    df = pd.DataFrame(results).sort_values(["subfolder", "image_name"]).reset_index(drop=True)
    summary = (
        df["source"].value_counts(dropna=False)
        .rename_axis("source")
        .reset_index(name="count")
    )
    summary["ratio"] = summary["count"] / max(len(df), 1)

    csv_path = OUT_ROOT / "per_image_results_v3.csv"
    xlsx_path = OUT_ROOT / "circle_anchor_test_report_v3.xlsx"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(xlsx_path, engine="openypxl") as writer:
        template_df.to_excel(writer, sheet_name="template_summary", index=False)
        # Save all_images_df, although it's not strictly necessary as image_paths are in results df.
        # If needed for separate tracking, uncomment and adjust column names for full path.
        # all_images_df.to_excel(writer, sheet_name="all_images_processed", index=False)
        df.to_excel(writer, sheet_name="per_image_results", index=False)
        summary.to_excel(writer, sheet_name="source_summary", index=False)

    elapsed = time.time() - t0
    print("\n===== 완료 =====")
    print(f"CSV : {csv_path}")
    print(f"XLSX: {xlsx_path}")
    print(f"Overlay 폴더: {overlay_dir}")
    print(f"총 시간: {elapsed/60:.1f}분")
    print(summary)


if __name__ == "__main__":
    main()

# =========================================================
# v3 수정 핵심
# =========================================================
# 1) 전체 이미지 처리 -> 하위 폴더별 랜덤 샘플 추출 제거
# 2) direct detect 제거
# 3) good template를 먼저 놓고, 그 근처 후보를 끼워 맞추는 template-driven 방식
# 4) matched_count >= 2 이면 corrected, 아니면 template_only
# 5) 최종 4점은 항상 template 구조(평행사변형)를 유지하도록 order + template 우선
# 6) QC 컬럼 추가: 4점 간의 너비, 높이, 대각선, 기울기, 면적, 중심점 등
# =========================================================


[1] good 케이스(notes=='ㅇ')로 template 생성 중...
good 케이스 수: 25
[2] 전체 이미지 목록 수집 중...
전체 이미지 수: 1,462
[3] template-driven 방식으로 anchor 4점 생성 중...


  0%|          | 0/1462 [00:00<?, ?it/s]

ValueError: No Excel writer 'openypxl'

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

# Define paths (consistent with the main script)
SAVEFILE_DIR = Path("/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2")
OUT_ROOT = SAVEFILE_DIR / "_circle_anchor_test_final"

# Define paths for the CSV files that should have been generated
template_summary_csv = OUT_ROOT / "template_summary.csv"
per_image_results_csv = OUT_ROOT / "per_image_results_v3.csv"
xlsx_path = OUT_ROOT / "circle_anchor_test_report_v3.xlsx"

try:
    # 1. Load template_df
    template_df = pd.read_csv(template_summary_csv)
    print(f"Loaded template summary from: {template_summary_csv}")

    # 2. Load result_df (which is 'df' in main function)
    df = pd.read_csv(per_image_results_csv)
    print(f"Loaded per image results from: {per_image_results_csv}")

    # 3. Recreate summary_df
    summary = (
        df["source"].value_counts(dropna=False)
        .rename_axis("source")
        .reset_index(name="count")
    )
    summary["ratio"] = summary["count"] / max(len(df), 1)
    print("Generated summary DataFrame.")

    # 4. Write to Excel
    print(f"Writing Excel report to: {xlsx_path}")
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        template_df.to_excel(writer, sheet_name="template_summary", index=False)
        df.to_excel(writer, sheet_name="per_image_results", index=False)
        summary.to_excel(writer, sheet_name="source_summary", index=False)
    print("Excel report generated successfully.")

except FileNotFoundError as e:
    print(f"Error: Required CSV file not found. Please ensure the full script has run successfully at least once to generate the CSVs. {e}")
except Exception as e:
    print(f"An error occurred during Excel generation: {e}")


Loaded template summary from: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_final/template_summary.csv
Loaded per image results from: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_final/per_image_results_v3.csv
Generated summary DataFrame.
Writing Excel report to: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0306-0402_2/_circle_anchor_test_final/circle_anchor_test_report_v3.xlsx
Excel report generated successfully.
